In [1]:
import os, sys, json

REPO = "/uscms/home/reshmar/nobackup/IDMe_Run3_Collab/CMSSW_13_0_13/src/iDMe"
AT   = os.path.join(REPO, "python_analysis/analysisTools")
TP   = os.path.join(REPO, "python_analysis/thesis_plots/electron_xclean")

os.environ["X509_USER_PROXY"] = os.path.expanduser("~/x509up_proxy.pem")
sys.path.append(AT)

from lpc_dask import make_lpc_client, ship_analysis_modules, expected_entries
from analysisTools import Analyzer
import coffea.util as util

sample_config = os.path.join(REPO, "python_analysis/configs/sample_configs/signal_2022_Mchi_ctau100_aEM.json")
cuts_config   = os.path.join(TP, "cut_configs/cuts_DSCB.py")
histos_config = os.path.join(TP, "histo_configs/Dxy_res_histo_for_fig20_pT.py")
outdir        = os.path.join(REPO, "python_analysis/coffea/AllInOne_2022")
os.makedirs(outdir, exist_ok=True)

cluster, client = make_lpc_client(n_workers=40)
print("dashboard:", cluster.dashboard_link)

try:
    # On a busy pool the first worker can take 10-15 minutes to appear (measured 822 s).
    # A long pause here is queueing, not a hang.
    client.wait_for_workers(1, timeout=1800)

    ship_analysis_modules(client, configs=[cuts_config, histos_config])

    analyzer = Analyzer(sample_config, histos_config, cuts_config, max_samples=-1)
    out = analyzer.process(execr="dask", dask_client=client)

    acc, metrics = out                   # savemetrics=True gives a 2-tuple
    print("entries:", metrics["entries"], "expected:", expected_entries(sample_config))

    util.save(out, os.path.join(outdir, "Sept1_inpTbins.coffea"))
finally:
    client.close(); cluster.close()      # must run even if the wait times out,
                                         # or the condor jobs sit holding slots

dashboard: http://localhost:8787/status
shipped mySchema.py
shipped analysisSubroutines.py
shipped corrections.py
shipped analysisTools.py
shipped cuts_DSCB.py
shipped Dxy_res_histo_for_fig20_pT.py
Running with systematics:  None
entries: 19894000 expected: 19894000#####] | 100% Completed | 33min 31.6s
